# Polars demo

An implementation *(via claude.ai)* of the nomis class using `polars` rather than `pandas`; note that the `nomis.nomis_data_lazy()` is *not* properly lazy on reading from the URL, because the nomic API doesnlt provide a required `Content-Length` header.

In [1]:
from nomis_api_polars_wrapper import NOMIS_CONFIG_POLARS

nomis = NOMIS_CONFIG_POLARS()

In [2]:
nomis_datasets = nomis.nomis_codes_datasets()
nomis_datasets.head()

agency,idx,name,description,source
str,str,str,str,str
"""NOMIS""","""NM_1_1""","""Jobseeker's Allowance with rat…","""Records the number of people c…","""jsa"""
"""NOMIS""","""NM_2_1""","""claimant count - age and durat…","""A quartery count of claimants …","""jsa"""
"""NOMIS""","""NM_4_1""","""Jobseeker's Allowance by age a…","""A monthly count of Jobseeker's…","""jsa"""
"""NOMIS""","""NM_5_1""","""claimant count denominators - …","""A midyear estimate of the work…","""jsa"""
"""NOMIS""","""NM_6_1""","""claimant count - occupation""","""A quarterly count of Jobseeke…","""jsa"""


## Census Datasets

Census data is available via `nomis`.

In [3]:
import polars as pl

census_datasets = nomis_datasets.filter(pl.col("source").str.contains("census_2021"))
census_datasets.head()

agency,idx,name,description,source
str,str,str,str,str
"""NOMIS""","""NM_2020_1""","""TS007A - Age by five-year age …","""This dataset provides Census 2…","""census_2021_ts"""
"""NOMIS""","""NM_2021_1""","""TS001 - Number of usual reside…","""This dataset provides Census 2…","""census_2021_ts"""
"""NOMIS""","""NM_2022_1""","""TS002 - Legal partnership stat…","""This dataset provides Census 2…","""census_2021_ts"""
"""NOMIS""","""NM_2023_1""","""TS003 - Household composition""","""This dataset provides Census 2…","""census_2021_ts"""
"""NOMIS""","""NM_2024_1""","""TS004 - Country of birth""","""This dataset provides Census 2…","""census_2021_ts"""


For example, the *Number of usual residents* gives us example population numbers.

In [4]:
USUAL_RESIDENTS_2021 = census_datasets.select(["idx", "name"]).filter(
    pl.col("name").str.contains("usual residents in households")
)[0, "idx"]

USUAL_RESIDENTS_2021

'NM_2021_1'

The results are provided for specified geographical areas. We can view the top level geographies for which the data is available.

In [5]:
# USUAL_RESIDENTS_2021 = "NM_2021_1"

# Get the geography codes for which this data is available
residents_geog_codes = nomis.nomis_codes_geog(USUAL_RESIDENTS_2021)
residents_geog_codes

agencyid,dataset,dimension,name,description,value
str,str,str,str,str,i64
"""NOMIS""","""""","""CL_2021_1_GEOGRAPHY""","""geography""","""England and Wales""",2092957703
"""NOMIS""","""""","""CL_2021_1_GEOGRAPHY""","""geography""","""England""",2092957699
"""NOMIS""","""""","""CL_2021_1_GEOGRAPHY""","""geography""","""Wales""",2092957700


For a particular geography code supported by a dataset, we can review the available datasets (lower level geographies) within that high-level geography:

In [ ]:
geog_code = residents_geog_codes.filter(pl.col("description") == "England")[0, "value"]

england_geog = nomis.nomis_codes_geog(USUAL_RESIDENTS_2021, geography=geog_code)
england_geog.head(10)

agencyid,dataset,dimension,name,description,value
str,str,str,str,str,str
"""NOMIS""","""""","""CL_2021_1_GEOGRAPHY""","""geography""","""England""","""2092957699"""
"""NOMIS""","""""","""CL_2021_1_GEOGRAPHY""","""geography""","""2021 output areas within Engla…","""2092957699TYPE150"""
"""NOMIS""","""""","""CL_2021_1_GEOGRAPHY""","""geography""","""2021 super output areas - lowe…","""2092957699TYPE151"""
"""NOMIS""","""""","""CL_2021_1_GEOGRAPHY""","""geography""","""2021 super output areas - midd…","""2092957699TYPE152"""
"""NOMIS""","""""","""CL_2021_1_GEOGRAPHY""","""geography""","""2022 wards within England""","""2092957699TYPE153"""
"""NOMIS""","""""","""CL_2021_1_GEOGRAPHY""","""geography""","""2022 local authorities: distri…","""2092957699TYPE154"""
"""NOMIS""","""""","""CL_2021_1_GEOGRAPHY""","""geography""","""2022 local authorities: counti…","""2092957699TYPE155"""
"""NOMIS""","""""","""CL_2021_1_GEOGRAPHY""","""geography""","""2021 Parliamentary constituenc…","""2092957699TYPE158"""
"""NOMIS""","""""","""CL_2021_1_GEOGRAPHY""","""geography""","""Sub Integrated Care Board Loca…","""2092957699TYPE164"""


Let's see if the data is available grouped by LA districts:

In [7]:
def pl_row_dict(pldf):
    return dict(zip(pldf.columns, pldf[0].row(0)))


en_la_districts = england_geog.filter(pl.col("description").str.contains("districts"))
en_la_districts = pl_row_dict(en_la_districts)
en_la_districts

{'agencyid': 'NOMIS',
 'dataset': '',
 'dimension': 'CL_2021_1_GEOGRAPHY',
 'name': 'geography',
 'description': '2022 local authorities: districts within England',
 'value': '2092957699TYPE154'}

See if we can find the geography code for the *Isle of Wight* LAD. We need to search in the *2022 local authorities: districts within England* geography, whose identifier is given by the `CL_2021_1_GEOGRAPHY` dimension `value`.

In [ ]:
iw = pl_row_dict(
    nomis.nomis_codes_geog(
        USUAL_RESIDENTS_2021,
        geography=en_la_districts["value"],
        search="Isle of Wight*",
    )
)

iw

{'agencyid': 'NOMIS',
 'dataset': '',
 'dimension': 'CL_2021_1_GEOGRAPHY',
 'name': 'geography',
 'description': 'Isle of Wight',
 'value': 645923034}

In [9]:
nomis.nomis_codes_geog(USUAL_RESIDENTS_2021, geography=iw["value"])

agencyid,dataset,dimension,name,description,value
str,str,str,str,str,str
"""NOMIS""","""""","""CL_2021_1_GEOGRAPHY""","""geography""","""Isle of Wight""","""645923034"""
"""NOMIS""","""""","""CL_2021_1_GEOGRAPHY""","""geography""","""2021 output areas within Isle …","""645923034TYPE150"""
"""NOMIS""","""""","""CL_2021_1_GEOGRAPHY""","""geography""","""2021 super output areas - lowe…","""645923034TYPE151"""
"""NOMIS""","""""","""CL_2021_1_GEOGRAPHY""","""geography""","""2021 super output areas - midd…","""645923034TYPE152"""
"""NOMIS""","""""","""CL_2021_1_GEOGRAPHY""","""geography""","""2022 wards within Isle of Wigh…","""645923034TYPE153"""


The rather cryptic `nomis.get_geo_code()` has a chase function that will also try to find corresponding records in child geographies in a rather more convenient way:

In [10]:
iw_ = nomis.get_geo_code(
    idx=USUAL_RESIDENTS_2021,
    value=en_la_districts["value"],
    search="Isle of Wight*",
    chase=True,
)
iw_

description,geog
str,str
"""Isle of Wight""","""645923034"""
"""2021 output areas within Isle …","""645923034TYPE150"""
"""2021 super output areas - lowe…","""645923034TYPE151"""
"""2021 super output areas - midd…","""645923034TYPE152"""
"""2022 wards within Isle of Wigh…","""645923034TYPE153"""


So how does the population break down for the Isle of Wight LAD?

In [11]:
cols = ["GEOGRAPHY_CODE", "C2021_RESTYPE_3_NAME", "MEASURES_NAME", "OBS_VALUE"]

lazy_data = nomis.nomis_data_lazy(USUAL_RESIDENTS_2021, geography=iw_[0, "geog"])

lazy_data.collect()

DATE,DATE_NAME,DATE_CODE,DATE_TYPE,DATE_TYPECODE,DATE_SORTORDER,GEOGRAPHY,GEOGRAPHY_NAME,GEOGRAPHY_CODE,GEOGRAPHY_TYPE,GEOGRAPHY_TYPECODE,GEOGRAPHY_SORTORDER,C2021_RESTYPE_3,C2021_RESTYPE_3_NAME,C2021_RESTYPE_3_CODE,C2021_RESTYPE_3_TYPE,C2021_RESTYPE_3_TYPECODE,C2021_RESTYPE_3_SORTORDER,MEASURES,MEASURES_NAME,OBS_VALUE,OBS_STATUS,OBS_STATUS_NAME,OBS_CONF,OBS_CONF_NAME,URN,RECORD_OFFSET,RECORD_COUNT,_Code
i64,i64,i64,str,i64,i64,i64,str,str,str,i64,i64,i64,str,str,str,i64,i64,i64,str,f64,str,str,str,str,str,i64,i64,str
2021,2021,2021,"""date""",0,0,645923034,"""Isle of Wight""","""E06000046""","""2022 local authorities: distri…",154,0,0,"""Total: All usual residents""","""0""","""Number of usual residents in h…",1000000,0,20100,"""Value""",140459.0,"""A""","""Normal Value""","""F""","""Free (free for publication)""","""Nm-2021d1d32336e0d645923034d0d…",0,6,"""NM_2021_1"""
2021,2021,2021,"""date""",0,0,645923034,"""Isle of Wight""","""E06000046""","""2022 local authorities: distri…",154,0,0,"""Total: All usual residents""","""0""","""Number of usual residents in h…",1000000,0,20301,"""Percent""",100.0,"""A""","""Normal Value""","""F""","""Free (free for publication)""","""Nm-2021d1d32336e0d645923034d0d…",1,6,"""NM_2021_1"""
2021,2021,2021,"""date""",0,0,645923034,"""Isle of Wight""","""E06000046""","""2022 local authorities: distri…",154,0,1,"""Lives in a household""","""_1""","""Number of usual residents in h…",1000000,1,20100,"""Value""",137516.0,"""A""","""Normal Value""","""F""","""Free (free for publication)""","""Nm-2021d1d32336e0d645923034d1d…",2,6,"""NM_2021_1"""
2021,2021,2021,"""date""",0,0,645923034,"""Isle of Wight""","""E06000046""","""2022 local authorities: distri…",154,0,1,"""Lives in a household""","""_1""","""Number of usual residents in h…",1000000,1,20301,"""Percent""",97.9,"""A""","""Normal Value""","""F""","""Free (free for publication)""","""Nm-2021d1d32336e0d645923034d1d…",3,6,"""NM_2021_1"""
2021,2021,2021,"""date""",0,0,645923034,"""Isle of Wight""","""E06000046""","""2022 local authorities: distri…",154,0,2,"""Lives in a communal establishm…","""_2""","""Number of usual residents in h…",1000000,2,20100,"""Value""",2943.0,"""A""","""Normal Value""","""F""","""Free (free for publication)""","""Nm-2021d1d32336e0d645923034d2d…",4,6,"""NM_2021_1"""
2021,2021,2021,"""date""",0,0,645923034,"""Isle of Wight""","""E06000046""","""2022 local authorities: distri…",154,0,2,"""Lives in a communal establishm…","""_2""","""Number of usual residents in h…",1000000,2,20301,"""Percent""",2.1,"""A""","""Normal Value""","""F""","""Free (free for publication)""","""Nm-2021d1d32336e0d645923034d2d…",5,6,"""NM_2021_1"""
